In [97]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [98]:
df = pd.read_csv(
    "/home/chintan/house_price_prediction/data/house_prices.csv",
    nrows=1000
)

In [99]:
def price_to_value(values):
    try:
        value = values.lower()
        if "cr" in  value:
            v = value.replace("cr","").strip()
            return float(v)*100
        v = value.replace("lac","").strip()
        if "lac" in  value:
            return float(value.replace("lac","").strip())
        return None
    except:
        return None

def car_parking_encoding(value):
    if type(value) == str and value and ("open" in value.lower() or "covered" in value.lower()):
        return 1
    return 0

48

def intTryParse(value):
    try:
        return int(value)
    except ValueError:
        return 0
    
def balcony_encoding(value):
    if type(value) != str:
        return 0
    # print(f"{value} {type(value)} {value} Before")
    if (type(value) == float or type(value) == int):
        return value
    return intTryParse(value)

def furnishing_encoding(value):
    if type(value) != str or "unfurnished" in value.lower():
        return 0
    if "semi-furnished" in value.lower() :
        return 0.5
    if "furnished" in value.lower() :
        return 1
    return 0


def overlooking_encoding(value):
    if type(value) != str:
        return None
    l = []
    if "road" in value.lower():
        l.append("road")
    if "garden" in value.lower() or "park" in value.lower():
            l.append("garden")
    if "pool" in value.lower():
            l.append("pool")   
    return ",".join(l)

In [100]:
df = df.drop(columns=["Index","Price (in rupees)","Description","Title","Transaction","facing","Society", "Super Area","Plot Area","Dimensions", "Status", "location"])
df = df.rename(columns={'Amount(in rupees)': 'amount', 'Car Parking': 'car_parking','Carpet Area': 'area'})
df["amount"] = df["amount"].apply(price_to_value)
df["car_parking"] = df["car_parking"].apply(car_parking_encoding)
df["Balcony"] = df["Balcony"].apply(balcony_encoding)
df["Furnishing"] = df["Furnishing"].apply(furnishing_encoding)
df["Bathroom"] = df["Bathroom"].apply(intTryParse)
df["overlooking"] = df["overlooking"].apply(overlooking_encoding)
df["area"] = df["area"].apply(lambda value: intTryParse(value.split(" ")[0]) if value != None and type(value) == str  else None)
df["Floor"] = df["Floor"].apply(lambda value: intTryParse(value.split(" ")[0]) if value != None and type(value) == str  else None)
df = df.dropna(subset=["amount"])
df = df.dropna(subset=["area"])
df = df.dropna(subset=["Ownership"])
df = df.dropna(subset=["Floor"])
df = pd.get_dummies(df, columns=["overlooking","Ownership"], dtype=int)
df = df.drop(df[df["area"] > 3000].index)


### Feature Scalling

In [101]:
mean_of_area = df["area"].mean()
mean_of_floor = df["Floor"].mean()
mean_of_bathroom = df["Bathroom"].mean()
mean_of_amount = df["amount"].mean()

variance_area = ((df["area"] - mean_of_area)**2).sum()/df.shape[0]
variance_floor = ((df["Floor"] - mean_of_floor)**2).sum()/df.shape[0]
variance_bathroom = ((df["Bathroom"] - mean_of_bathroom)**2).sum()/df.shape[0]
variance_amount = ((df["amount"] - mean_of_amount)**2).sum()/df.shape[0]

standard_deviation_area = variance_area**0.5
standard_deviation_floor = variance_floor**0.5
standard_deviation_bathroom = variance_bathroom**0.5
standard_deviation_amount = variance_amount**0.5

scaled_area = (df["area"] - mean_of_area) / standard_deviation_area
scaled_floor = (df["Floor"] - mean_of_floor) / standard_deviation_floor
scaled_bathroom = (df["Bathroom"] - mean_of_bathroom) / standard_deviation_bathroom
scaled_amount = (df["amount"] - mean_of_amount) / standard_deviation_amount

df["area"] = scaled_area
df["Floor"] = scaled_floor
df["Bathroom"] = scaled_bathroom
df["amount"] = scaled_amount


### Train and Test Split

In [102]:
df_train = df.sample(frac=0.8,random_state=42)
df_test = df.drop(df_train.index)

### Train and Test X and Y values

In [103]:
Y_train = df_train["amount"]
Y_test = df_test["amount"]
X_train = df_train.drop("amount",axis=1)
X_test = df_test.drop("amount",axis=1)

In [104]:
# df.head(n=50)
# Y_train

### Remove Outliners 

In [105]:

# Q1 = df["amount"].quantile(0.30)
# Q3 = df["amount"].quantile(0.80)

# IQR = Q3 - Q1

# lower = Q1 - 1.5 * IQR
# upper = Q3 + 1.5 * IQR

# outliers = df[
#     (df["amount"] < lower) |
#     (df["amount"] > upper)
# ]



In [106]:
weights = np.zeros(X_train.shape[1])

bias = 0

learning_rate = 0.01

In [107]:

for i in range(2000):
    y_predicted = (X_train * weights).sum(axis=1) + bias
    errors = (Y_train - y_predicted).to_numpy()
    mean_squered_error = ((errors**2).sum()/y_predicted.shape[0])**0.5
    gradients = (2 / X_train.shape[0]) * (errors[:, np.newaxis] * X_train).sum(axis=0)
    bias_gradient = (2 / X_train.shape[0]) * errors.sum()
    weights = weights + learning_rate * gradients
    bias = bias + learning_rate * bias_gradient
    # print(errors[:5],"==========>> Weights",i)
    # print(mean_squered_error,"==========>> MSE",i)


In [108]:
y_test_predicted = (X_test * weights).sum(axis=1) + bias



print((y_test_predicted * standard_deviation_amount) + mean_of_amount) 
# print(y_test_predicted) 


2      152.627496
18     121.422612
26     -10.253693
28     165.331694
34      11.758855
          ...    
960    103.120155
963    104.542761
972    113.465275
983    297.552191
988    123.359999
Length: 112, dtype: float64


In [109]:


print((Y_test * standard_deviation_amount) + mean_of_amount) 
print(Y_test[:5] ) 
type(Y_test)



2      140.0
18      90.0
26      24.0
28     175.0
34      54.0
       ...  
960    104.0
963    100.0
972    107.0
983    145.0
988    110.0
Name: amount, Length: 112, dtype: float64
2     0.182666
18   -0.327747
26   -1.001492
28    0.539955
34   -0.695244
Name: amount, dtype: float64


pandas.Series

### Show graph

In [ ]:
x = np.arange(len(Y_test[:20]))

plt.figure(figsize=(10, 5))
plt.scatter(x, Y_test[:20].values, color='blue', label='Actual')
plt.scatter(x, y_test_predicted[:20].values, color='red', label='Predicted')
plt.xlabel('Sample index')
plt.ylabel('Amount (scaled)')
plt.title('Actual vs Predicted')
plt.legend()
plt.show()



SyntaxError: invalid syntax (3469710783.py, line 13)